# Notebook 00 — Setup and IFC Exploration

Verifies the environment, installs dependencies, and explores the IFC reference model schema.

**No GPU required.** Run on CPU runtime.

> **Run Cell 1 first every session.** It clones/updates the repo, installs packages, and sets the working directory.

In [ ]:
# ── Cell 1: Clone, install, set working directory ─────────────────────────────
import os, sys

REPO = 'ifc-graphrag-dt'

# Clone or update
if os.path.exists(REPO):
    !cd {REPO} && git pull --quiet
    print('Repository updated.')
else:
    !git clone https://github.com/aiwithprashant/ifc-graphrag-dt.git
    print('Repository cloned.')

# Install dependencies
!bash {REPO}/colab_setup.sh

# CRITICAL: change working directory into the repo
# This makes all relative paths (benchmark/prompts/, outputs/, etc.) work correctly
os.chdir(REPO)
print(f'Working directory: {os.getcwd()}')

# Add to Python path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print('\nSetup complete.')

In [ ]:
# ── Cell 2: Verify imports ────────────────────────────────────────────────────
import networkx as nx
import numpy as np
import json
from pathlib import Path

from benchmark.dtah_bench import DTAHBench
from evaluation.metrics.kcs_dt import KCSDTScorer
from evaluation.metrics.ggs import GGSScorer
from pipeline.layer1_retriever.khop_traversal import KHopTraversal

print('All core imports successful.')

In [ ]:
# ── Cell 3: DTAH-Bench overview ───────────────────────────────────────────────
bench = DTAHBench()   # uses default prompts_dir='benchmark/prompts' — works because cwd is repo root
stats = bench.stats()
print('DTAH-Bench statistics:')
for k, v in stats.items():
    print(f'  {k}: {v}')

print('\nSample Tier 1 prompt:')
t1 = bench.load_tier(1)
print(json.dumps(t1[0], indent=2))

In [ ]:
# ── Cell 4: Prompt distribution ───────────────────────────────────────────────
from collections import Counter
import matplotlib
matplotlib.use('Agg')  # headless-safe backend
import matplotlib.pyplot as plt

os.makedirs('outputs/figures', exist_ok=True)

all_prompts = bench.load_all()
tier_counts   = Counter(p['tier'] for p in all_prompts)
domain_counts = Counter(p['domain'] for p in all_prompts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar([f'Tier {t}' for t in sorted(tier_counts)],
            [tier_counts[t] for t in sorted(tier_counts)],
            color=['#2E5FA3','#1D9E75','#D85A30'])
axes[0].set_title('Prompts per Tier')
axes[0].set_ylabel('Count')

axes[1].bar(list(domain_counts.keys()), list(domain_counts.values()),
            color=['#2E5FA3','#1D9E75','#D85A30'])
axes[1].set_title('Prompts per Domain')

plt.tight_layout()
plt.savefig('outputs/figures/00_prompt_distribution.png', dpi=150)
plt.show()
print(f'Total prompts: {len(all_prompts)}')
print('Figure saved: outputs/figures/00_prompt_distribution.png')

In [ ]:
# ── Cell 5: Download IFC reference model ─────────────────────────────────────
import urllib.request

IFC_DIR    = 'benchmark/ifc_reference_models'
DUPLEX_URL = 'https://github.com/buildingSMART/Sample-Test-Files/raw/master/IFC%202x3/Duplex%20Apartment/Duplex_A_20110907.ifc'
DUPLEX_PATH = f'{IFC_DIR}/duplex.ifc'

os.makedirs(IFC_DIR, exist_ok=True)

if not os.path.exists(DUPLEX_PATH):
    print('Downloading Duplex IFC reference model (~1.2 MB)...')
    try:
        urllib.request.urlretrieve(DUPLEX_URL, DUPLEX_PATH)
        print(f'Downloaded: {os.path.getsize(DUPLEX_PATH)/1024:.1f} KB')
    except Exception as e:
        # Fallback: use wget
        print(f'urllib failed ({e}), trying wget...')
        !wget -q -O {DUPLEX_PATH} "{DUPLEX_URL}"
        print(f'Downloaded via wget: {os.path.getsize(DUPLEX_PATH)/1024:.1f} KB')
else:
    print(f'Already exists: {DUPLEX_PATH} ({os.path.getsize(DUPLEX_PATH)/1024:.1f} KB)')

In [ ]:
# ── Cell 6: Explore IFC schema with ifcopenshell ──────────────────────────────
import ifcopenshell

model = ifcopenshell.open(DUPLEX_PATH)
print(f'IFC schema: {model.schema}')

type_counts = {}
for entity in model:
    t = entity.is_a()
    type_counts[t] = type_counts.get(t, 0) + 1

sorted_types = sorted(type_counts.items(), key=lambda x: -x[1])
print(f'\nTotal entities:       {sum(type_counts.values())}')
print(f'Unique entity types:  {len(type_counts)}')
print('\nTop 20 entity types:')
for t, n in sorted_types[:20]:
    print(f'  {t:<45} {n}')

In [ ]:
# ── Cell 7: Explore IFC relations tracked by the pipeline ─────────────────────
from pipeline.layer1_retriever.ifc_graph_builder import IFC_RELATION_CATEGORIES

print('IFC relations tracked by IFC-GraphRAG-DT pipeline:')
print(f'{"Relation Type":<48} {"Semantic Category":<25} Instances in model')
print('-' * 90)
for rel, cat in IFC_RELATION_CATEGORIES.items():
    count = len(model.by_type(rel))
    print(f'{rel:<48} {cat:<25} {count}')

In [ ]:
# ── Cell 8: KCS-DT smoke test ─────────────────────────────────────────────────
scorer = KCSDTScorer()
gt = {
    'entities':    [{'ifc_type': 'IfcPump'}, {'ifc_type': 'IfcPipeSegment'}, {'ifc_type': 'IfcValve'}],
    'relations':   [{'type': 'IfcRelConnectsPortToElement', 'from': 'IfcPump.OutletPort', 'to': 'IfcPipeSegment'},
                    {'type': 'IfcRelConnects', 'from': 'IfcPipeSegment', 'to': 'IfcValve'}],
    'attributes':  {'pump_0': {'material': 'cast_iron'}},
    'containment': [{'entity': 'IfcPump', 'container': 'IfcSpace'}],
    'connectivity':[{'from': 'IfcPump.OutletPort', 'to': 'IfcPipeSegment'}]
}

perfect = scorer.score(gt, gt, prompt_id='smoke-test')
print(f'KCS-DT perfect score: {perfect}')
assert abs(perfect.total - 1.0) < 1e-4, 'Expected 1.0'

empty_pred = {'entities':[],'relations':[],'attributes':{},'containment':[],'connectivity':[]}
empty = scorer.score(empty_pred, gt, prompt_id='empty-test')
print(f'KCS-DT empty pred score: {empty}')
assert empty.entity == 0.0, 'Expected 0.0 entity score'

print('\nKCS-DT smoke test PASSED.')

In [ ]:
# ── Cell 9: GGS smoke test ────────────────────────────────────────────────────
ggs_scorer = GGSScorer()

G = nx.DiGraph()
G.add_node('A', ifc_type='IfcPump',        name='P-01')
G.add_node('B', ifc_type='IfcPipeSegment', name='Pipe-01')
G.add_node('C', ifc_type='IfcValve',       name='V-01')
G.add_edge('A', 'B', relation_type='IfcRelConnectsPortToElement')
G.add_edge('B', 'C', relation_type='IfcRelConnects')

ggs = ggs_scorer.score(G, G, prompt_id='smoke-test')
print(f'GGS perfect score: {ggs}')
assert abs(ggs.node_recall - 1.0) < 1e-4
assert abs(ggs.edge_recall - 1.0) < 1e-4

print('\nGGS smoke test PASSED.')
print('\n' + '='*55)
print('All checks passed. Ready for Notebook 01.')
print('='*55)